In [1]:
# =========================
# 1) Load Document
# =========================
import os
from langchain_community.document_loaders import PyPDFLoader
current_dir = os.path.dirname(os.path.abspath("__file__"))
file_path = os.path.join(current_dir, "TroubleshootingGuide-FrontlineLAN.pdf")
loader = PyPDFLoader(file_path)
documents = loader.load()
documents[0]

c:\Users\OI.naoto\prodapt-training\network-eng-assistant\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:26: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


Document(metadata={'producer': 'Adobe PDF Library 8.0', 'creator': 'Adobe InDesign CS3 (5.0.2)', 'creationdate': '2008-04-25T15:49:03-07:00', 'author': 'Fluke Networks', 'moddate': '2008-09-26T13:12:03-04:00', 'title': 'Frontline LAN Troubleshooting Guide', 'trapped': '/False', 'source': 'c:\\Users\\OI.naoto\\prodapt-training\\network-eng-assistant\\scripts\\TroubleshootingGuide-FrontlineLAN.pdf', 'total_pages': 115, 'page': 0, 'page_label': 'F2'}, page_content='NETWORK SUPER VISION\nFrontline LAN\nTroubleshooting Guide')

In [2]:
# Load environment variables from .env
from dotenv import load_dotenv
load_dotenv()

True

```
{
  "chunk_id": "TroubleshootingGuide-FrontlineLAN#b12#c3",
  "page_start": 8,
  "page_end": 9,
  "section_path": ["Eight steps to successful troubleshooting"],
  "raw_text": "…原文…",
  "tags": {
    "phase": "intro | identification | localization | analysis | action | verification",
  },
  "safety": {
    "is_safety": true,
    "reason": "laser_eye | backup_rollback | service_impact | security | other | none",
    "risk": "low | medium | high | none"
  },
}
```

In [3]:
# =========================
# 2) Spliting per subtitle
# =========================
TOC = [
  {"title": "Abstract", "part": "intro", "page_start": 2, "page_end": 2},
  {"title": "The best method", "part": "intro", "page_start": 3, "page_end": 4},
  {"title": "The process", "part": "intro", "page_start": 4, "page_end": 7},
  {"title": "Eight steps to successful troubleshooting", "part": "intro", "page_start": 8, "page_end": 15},
  {"title": "Troubleshooting copper media", "part": "physical", "page_start": 16, "page_end": 20},
  {"title": "Copper cable tests", "part": "physical", "page_start": 21, "page_end": 35},
  {"title": "Troubleshooting fiber optic media", "part": "physical", "page_start": 36, "page_end": 41},
  {"title": "Fiber optic cable tests", "part": "physical", "page_start": 41, "page_end": 46},
  {"title": "Troubleshooting common network user complaints", "part": "network", "page_start": 47, "page_end": 48},
  {"title": "Complaint: Can’t connect", "part": "network", "page_start": 48, "page_end": 52},
  {"title": "Complaint: Connection drops", "part": "network", "page_start": 53, "page_end": 57},
  {"title": "Complaint: Network is slow", "part": "network", "page_start": 57, "page_end": 62},
  {"title": "Typical switched network problems", "part": "switches", "page_start": 62, "page_end": 64},
  {"title": "Isolating the problem", "part": "switches", "page_start": 64, "page_end": 65},
  {"title": "Switch troubleshooting techniques", "part": "switches", "page_start": 65, "page_end": 67},
  {"title": "Method 1: Access the switch console", "part": "switches", "page_start": 67, "page_end": 69},
  {"title": "Method 2: Connect to an unused port", "part": "switches", "page_start": 70, "page_end": 77},
  {"title": "Method 3: Configure a mirror or span port", "part": "switches", "page_start": 77, "page_end": 84},
  {"title": "Method 4: Connect to a tagged or trunk port", "part": "switches", "page_start": 84, "page_end": 85},
  {"title": "Method 5: Insert a hub into the link", "part": "switches", "page_start": 86, "page_end": 90},
  {"title": "Method 6: Place the tester in series", "part": "switches", "page_start": 90, "page_end": 91},
  {"title": "Method 7: Place a Tap inline on a link", "part": "switches", "page_start": 91, "page_end": 97},
  {"title": "Method 8: Use SNMP-based management", "part": "switches", "page_start": 98, "page_end": 106},
  {"title": "Method 9: Use flow technology", "part": "switches", "page_start": 106, "page_end": 109},
  {"title": "Method 10: Set up a syslog server", "part": "switches", "page_start": 109, "page_end": 110},
  {"title": "Method 11: Use the server (host) resources", "part": "switches", "page_start": 110, "page_end": 112},
  {"title": "Method 12: Use a combination of the methods", "part": "switches", "page_start": 112, "page_end": 112},
  {"title": "Conclusion", "part": "conclusion", "page_start": 113, "page_end": 114},
]

In [4]:
from langchain_core.documents import Document

def build_blocks_from_toc(docs, toc_ranges):
    blocks = []
    for r in toc_ranges:
        print(r)
        ps, pe = r["page_start"], r["page_end"]
        texts = []
        for p in range(ps, pe + 1):
            t = (docs[p].page_content or "").strip()
            if t:
                texts.append(t)
        joined = "\n\n".join(texts).strip()
        if not joined:
            continue

        title = r["title"]
        part = r.get("part", "unknown")

        blocks.append(Document(
            page_content=joined,
            metadata={
                "page_start": ps,
                "page_end": pe,
                "section_title": title,
                "part": part,
                "block_id": f"#p{ps}-{pe}#{title}",
            }
        ))
    return blocks

blocks = build_blocks_from_toc(documents, TOC)
len(blocks), blocks[0].metadata, blocks[0].page_content

{'title': 'Abstract', 'part': 'intro', 'page_start': 2, 'page_end': 2}
{'title': 'The best method', 'part': 'intro', 'page_start': 3, 'page_end': 4}
{'title': 'The process', 'part': 'intro', 'page_start': 4, 'page_end': 7}
{'title': 'Eight steps to successful troubleshooting', 'part': 'intro', 'page_start': 8, 'page_end': 15}
{'title': 'Troubleshooting copper media', 'part': 'physical', 'page_start': 16, 'page_end': 20}
{'title': 'Copper cable tests', 'part': 'physical', 'page_start': 21, 'page_end': 35}
{'title': 'Troubleshooting fiber optic media', 'part': 'physical', 'page_start': 36, 'page_end': 41}
{'title': 'Fiber optic cable tests', 'part': 'physical', 'page_start': 41, 'page_end': 46}
{'title': 'Troubleshooting common network user complaints', 'part': 'network', 'page_start': 47, 'page_end': 48}
{'title': 'Complaint: Can’t connect', 'part': 'network', 'page_start': 48, 'page_end': 52}
{'title': 'Complaint: Connection drops', 'part': 'network', 'page_start': 53, 'page_end': 57}


(28,
 {'page_start': 2,
  'page_end': 2,
  'section_title': 'Abstract',
  'part': 'intro',
  'block_id': '#p2-2#Abstract'},
 'Fluke Networks 2\nFrontline LAN troubleshooting guide\nAbstract\nLocal area networks are integral to the operation of many  \nbusinesses today. Network engineers and network technicians  \nhave taken on the vital role of keeping these business-critical  \nnetworks up and running. This guide provides these frontline \nnetwork troubleshooters with practical advice on how to maintain \nLANs and solve common problems.\nA local area network (LAN) is comprised of many elements: printers, \nmonitors, PCs, IP phones, servers, storage hardware, networking \nequipment, security software, network applications, enterprise \napplications, office productivity applications, and more. In this \nguide, we will focus on layers 1 and 2 – the physical cable plant \nand switches. Network cabling and switches are the foundation of \ntoday’s local area networks.\nThis guide begins wit

In [5]:
import pandas as pd

def summarize_blocks(blocks):
    rows = []
    for b in blocks:
        text = b.page_content or ""
        rows.append({
            "title": b.metadata["section_title"],
            "part": b.metadata.get("part"),
            "page_start": b.metadata["page_start"],
            "page_end": b.metadata["page_end"],
            "pages": b.metadata["page_end"] - b.metadata["page_start"] + 1,
            "chars": len(text),
            "chars_per_page": round(len(text) / max(1, (b.metadata["page_end"] - b.metadata["page_start"] + 1)), 1),
        })
    return pd.DataFrame(rows)

df = summarize_blocks(blocks)
df

,title,part,page_start,page_end,pages,chars,chars_per_page
0,Abstract,intro,2,2,1,1518,1518.0
1,The best method,intro,3,4,2,3387,1693.5
2,The process,intro,4,7,4,6594,1648.5
3,Eight steps to successful troubleshooting,intro,8,15,8,13528,1691.0
4,Troubleshooting copper media,physical,16,20,5,9322,1864.4
5,Copper cable tests,physical,21,35,15,22301,1486.7
6,Troubleshooting fiber optic media,physical,36,41,6,9927,1654.5
7,Fiber optic cable tests,physical,41,46,6,7429,1238.2
8,Troubleshooting common network user complaints,network,47,48,2,3042,1521.0
9,Complaint: Can’t connect,network,48,52,5,8225,1645.0


In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from copy import deepcopy

# 分割ポリシー
SPLIT_THRESHOLD = 2000     # 8,000字超だけ分割
CHUNK_SIZE = 2500
CHUNK_OVERLAP = 200

# 「1行空き」を最優先にする splitter
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",      # ← 空行（段落）
        "\n- ",      # 箇条書き（任意）
        "\n• ",      # 箇条書き（任意）
        "\n",        # 改行
        ". ",        # 文
        " ",         # 単語
        ""           # 最終手段
    ],
)

separated_blocks = []

for b in blocks:
    text = b.page_content or ""

    base_meta = deepcopy(b.metadata)  # ← ここ重要（参照を切る）

    if len(text) <= SPLIT_THRESHOLD:
        meta = deepcopy(base_meta)
        meta["chunk_id"] = f"{base_meta['block_id']}#c1"
        separated_blocks.append(Document(page_content=text, metadata=meta))
        continue

    parts = splitter.split_documents([Document(page_content=text, metadata=base_meta)])

    for i, p in enumerate(parts, start=1):
        meta = deepcopy(p.metadata)   # 念のため参照切り
        meta["chunk_id"] = f"{base_meta['block_id']}#c{i}"
        separated_blocks.append(Document(page_content=p.page_content, metadata=meta))


len(separated_blocks), separated_blocks[34].metadata, separated_blocks[35].metadata

(123,
 {'page_start': 36,
  'page_end': 41,
  'section_title': 'Troubleshooting fiber optic media',
  'part': 'physical',
  'block_id': '#p36-41#Troubleshooting fiber optic media',
  'chunk_id': '#p36-41#Troubleshooting fiber optic media#c1'},
 {'page_start': 36,
  'page_end': 41,
  'section_title': 'Troubleshooting fiber optic media',
  'part': 'physical',
  'block_id': '#p36-41#Troubleshooting fiber optic media',
  'chunk_id': '#p36-41#Troubleshooting fiber optic media#c2'})

In [7]:
df = summarize_blocks(separated_blocks)
df

,title,part,page_start,page_end,pages,chars,chars_per_page
0,Abstract,intro,2,2,1,1518,1518.0
1,The best method,intro,3,4,2,1887,943.5
2,The best method,intro,3,4,2,1498,749.0
3,The process,intro,4,7,4,1498,374.5
4,The process,intro,4,7,4,1910,477.5
...,...,...,...,...,...,...,...
118,Method 11: Use the server (host) resources,switches,110,112,3,1673,557.7
119,Method 11: Use the server (host) resources,switches,110,112,3,1444,481.3
120,Method 12: Use a combination of the methods,switches,112,112,1,1444,1444.0
121,Conclusion,conclusion,113,114,2,1825,912.5


In [3]:
from langchain_openai import OpenAIEmbeddings
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [9]:
from langchain_qdrant import QdrantVectorStore

COLLECTION = "frontline_lan_metadata"

vs = QdrantVectorStore.from_documents(
    documents=separated_blocks,
    embedding=embeddings,
    collection_name=COLLECTION,
    path="../qdrant_store",
    force_recreate=True
)



In [10]:
query = "ネットワークが遅い。まず何を確認すべき？"

results = vs.similarity_search(
    query,
    k=5
)

for r in results:
    print(r.metadata)
    print(r.page_content[:300], "\n---\n")

{'page_start': 53, 'page_end': 57, 'section_title': 'Complaint: Connection drops', 'part': 'network', 'block_id': '#p53-57#Complaint: Connection drops', 'chunk_id': '#p53-57#Complaint: Connection drops#c2', '_id': '8f0dee3b86db45f7aa013697685e1ee6', '_collection_name': 'frontline_lan_metadata'}
www .flukenetworks .com57
tests continuously in order to detect the probable location of  
fluctuating or intermittent problems. For faster troubleshooting  
once a remote location is identified as being suspect, use network 
management to query the suspect infrastructure device and the 
infrastruct 
---

{'page_start': 53, 'page_end': 57, 'section_title': 'Complaint: Connection drops', 'part': 'network', 'block_id': '#p53-57#Complaint: Connection drops', 'chunk_id': '#p53-57#Complaint: Connection drops#c5', '_id': 'e0d84c0a9e84401499739601b71df937', '_collection_name': 'frontline_lan_metadata'}
www .flukenetworks .com57
tests continuously in order to detect the probable location of  
fluctuatin

In [12]:
from qdrant_client.http.models import Filter, FieldCondition, MatchValue

results = vs.similarity_search(
    "ネットワークが遅い。まず何を確認すべき？",
    k=5,
    filter=Filter(
        should=[
            FieldCondition(
                key="part",
                match=MatchValue(
                    value="network"
                ),
            ),
        ]
    )
)

for r in results:
    print(r.metadata)
    print(r.page_content[:300], "\n---\n")